In [31]:
#import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
#from itertools import product
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

## where trained models are saved (project-root models/ folder)
MODEL_DIR = Path("../models").resolve()
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df_2026 = pd.read_csv("../data/df_2026_final.csv").assign(season=2026)

# make 2026 consistent with 2024/2025: drop the 2026-only columns and use the same column order
# (2026 lacks the defensive columns, which are ~all NaN in 2025 and unused -> filled with NaN)
df_2026 = df_2026.drop(columns=["pro_team_id", "season_type"], errors="ignore")
df_2026 = df_2026.reindex(columns=df_2024.columns)
df_2026 = df_2026.astype(df_2024.dtypes.to_dict(), errors="ignore")

combined = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)

In [33]:
#Sort chronologically by player
combined = combined.sort_values(['player_id', 'season', 'week']).reset_index(drop=True)

#Create target (Next week's fantasy score) and next week's context via shift(-1)
#FIX: group by player AND season so a season's last game doesn't take next season's week 1 as its target
g = combined.groupby(['player_id', 'season'])
combined['target_next_week_fantasy'] = g['fantasy_points'].shift(-1)
combined['next_week_opponent'] = g['opponent'].shift(-1)
combined['next_week_atVs'] = g['atVs'].shift(-1)
combined['next_week_projected'] = g['projected_points'].shift(-1)   # ESPN's projection for the game being predicted
for col in ['roof', 'surface', 'temp', 'wind', 'location']:
    combined[f'next_week_{col}'] = g[col].shift(-1)
# next_week_opp_def_rank removed: opp_def_rank is a season-end value, so it leaks future games

In [34]:
#Compute cumulative career and season-to-date stats up to current row (Week N)
stats_to_accumulate = [
    'fantasy_points', 'receptions', 'receivingTargets', 'receivingYards', 
    'receivingTouchdowns', 'rushingAttempts', 'rushingYards', 'rushingTouchdowns',
    'completions', 'passingAttempts', 'passingYards', 'passingTouchdowns', 
    'interceptions', 'fumblesLost'
]

for col in stats_to_accumulate:
    # 1. Career cumulative mean (Extract values to bypass MultiIndex mismatch)
    combined[f'{col}_cum_career_avg'] = (
        combined.groupby('player_id')[col]
        .expanding()
        .mean()
        .values
    )
    
    # 2. Season-to-date cumulative mean
    combined[f'{col}_season_avg'] = (
        combined.groupby(['player_id', 'season'])[col]
        .expanding()
        .mean()
        .values
    )

for col in ['fantasy_points', 'receivingTargets', 'rushingAttempts', 'passingAttempts']:
    combined[f'{col}_last3'] = (
        combined.groupby(['player_id', 'season'])[col]
        .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

In [35]:
# combined = combined[(combined['season'] == 2024) | (combined['season'] == 2025) & (combined['week'] < 18)]

In [36]:
categorical_cols = ['position', 'next_week_opponent', 'next_week_atVs', 'next_week_surface',
                    'next_week_roof', 'next_week_location']
for col in categorical_cols:
    if col in combined.columns:
        combined[col] = combined[col].astype('category')

In [37]:
irrelevant_columns = [
    'Unnamed: 0', 'player_id', 'player', 'team', 'opponent', 'date',
    'eventId', 'game_id', 'gameday', 'gametime', 'stadium', 'result',
    'score', 'fantasy_points', 'target_next_week_fantasy', 'espn',
    '_set', 'season',
    'med_proj_pts',
    'projected_points',
    'atVs', 'roof', 'surface', 'temp', 'wind', 'location',
    'opp_def_rank', 'team_pass_rank', 'team_rush_rank', 'next_week_projected' 
]

relevant_columns = [col for col in combined.columns if col not in irrelevant_columns]

In [38]:
relevant_columns

['position',
 'week',
 'receptions',
 'receivingTargets',
 'receivingYards',
 'yardsPerReception',
 'receivingTouchdowns',
 'longReception',
 'rushingAttempts',
 'rushingYards',
 'yardsPerRushAttempt',
 'longRushing',
 'rushingTouchdowns',
 'fumbles',
 'fumblesLost',
 'fumblesForced',
 'kicksBlocked',
 'completions',
 'passingAttempts',
 'passingYards',
 'completionPct',
 'yardsPerPassAttempt',
 'passingTouchdowns',
 'interceptions',
 'longPassing',
 'sacks',
 'QBRating',
 'adjQBR',
 'longFieldGoalMade',
 'fieldGoalPct',
 'fieldGoalsMadeYardsAverage',
 'totalKickingPoints',
 'totalTackles',
 'soloTackles',
 'assistTackles',
 'stuffs',
 'stuffYards',
 'fumblesRecovered',
 'interceptionYards',
 'avgInterceptionYards',
 'interceptionTouchdowns',
 'longInterception',
 'passesDefended',
 'fieldGoalsMade1_19',
 'fieldGoalAttempts1_19',
 'fieldGoalsMade20_29',
 'fieldGoalAttempts20_29',
 'fieldGoalsMade30_39',
 'fieldGoalAttempts30_39',
 'fieldGoalsMade40_49',
 'fieldGoalAttempts40_49',
 'fie

In [39]:
target = 'target_next_week_fantasy'

In [40]:
train_data = combined[
    (combined['season'] == 2024) |
    ((combined['season'] == 2025) & (combined['week'] < 10))
]
val_data = combined[
    (combined['season'] == 2025) & (combined['week'].between(10, 18))
]
test_data = combined[
    (combined['season'] == 2026) & (combined['week'] < 11)
]
print(len(train_data), len(val_data), len(test_data))

# rebuild from scratch so stale cells can't interfere
X_train, y_train = train_data[relevant_columns], train_data[target]
X_val,   y_val   = val_data[relevant_columns],   val_data[target]
X_test,  y_test  = test_data[relevant_columns],  test_data[target]

# diagnostic: how many bad labels are there?
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(name, 'NaN:', y.isna().sum(), 'inf:', np.isinf(y).sum(), 'rows:', len(y))

# keep only rows with a finite target
def clean(X, y):
    m = y.notna() & np.isfinite(y)
    return X[m], y[m]

X_train, y_train = clean(X_train, y_train)
X_val,   y_val   = clean(X_val,   y_val)
X_test,  y_test  = clean(X_test,  y_test)

print(len(X_train), len(X_val), len(X_test))

11286 3808 4155
train NaN: 731 inf: 0 rows: 11286
val NaN: 625 inf: 0 rows: 3808
test NaN: 76 inf: 0 rows: 4155
10555 3183 4079


In [41]:
model_path = MODEL_DIR / "xgb_global.json"   # where this model is saved after training
model = xgb.XGBRegressor(
    n_estimators=10000, #Higher is better
    learning_rate=0.005, #lower is better
    max_depth=5, #higher is better
    min_child_weight=3, #mid:1-10
    subsample=0.7, #mid:0-1
    colsample_bytree=0.7, #mid:0-1
    random_state=12, #seed
    enable_categorical=True,
    early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
)

In [42]:
# model = xgb.XGBRegressor(
#     n_estimators=10000, #Higher is better
#     learning_rate=0.01, #lower is better
#     max_depth=8, #higher is better
#     min_child_weight=5, #mid:1-10
#     subsample=0.5, #mid:0-1
#     colsample_bytree=0.5, #mid:0-1
#     random_state=42, #seed
#     enable_categorical=True,
#     early_stopping_rounds=50      # stops when validation stops improving (old run overfit after ~50 trees)
# ) 806

In [43]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)
model.save_model(str(model_path))
print(f"saved model to {model_path}")

## reload later without retraining:
# model = xgb.XGBRegressor()
# model.load_model(str(model_path))

[0]	validation_0-rmse:7.58291


[50]	validation_0-rmse:6.92229
[100]	validation_0-rmse:6.48595
[150]	validation_0-rmse:6.20332
[200]	validation_0-rmse:6.02763
[250]	validation_0-rmse:5.91943
[300]	validation_0-rmse:5.85528
[350]	validation_0-rmse:5.82031
[400]	validation_0-rmse:5.80026
[450]	validation_0-rmse:5.78990
[500]	validation_0-rmse:5.78730
[550]	validation_0-rmse:5.78664
[583]	validation_0-rmse:5.78786


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [47]:
test_results = combined.loc[X_test.index].copy()
test_results['model_predicted_score'] = model.predict(X_test)
test_results['actual_score'] = y_test                      # next week's actual
test_results['espn_projection'] = test_results['next_week_projected']  # ESPN's projection for that same game

test_results['model_error'] = test_results['actual_score'] - test_results['model_predicted_score']
test_results['model_abs_error'] = test_results['model_error'].abs()
test_results['espn_error'] = test_results['actual_score'] - test_results['espn_projection']
test_results['espn_abs_error'] = test_results['espn_error'].abs()

# only compare rows where ESPN has a projection
cmp = test_results.dropna(subset=['espn_projection']).copy()

cmp['winner'] = np.select(
    [cmp['model_abs_error'] < cmp['espn_abs_error'],
     cmp['model_abs_error'] > cmp['espn_abs_error']],
    ['Model', 'ESPN'],
    default='Tie'
)

# overall metrics
for name, pred in [('Model', cmp['model_predicted_score']), ('ESPN', cmp['espn_projection'])]:
    print(f"{name}: MAE={mean_absolute_error(cmp['actual_score'], pred):.3f}  "
          f"RMSE={np.sqrt(mean_squared_error(cmp['actual_score'], pred)):.3f}  "
          f"R2={r2_score(cmp['actual_score'], pred):.3f}")

# biggest model misses
big_misses = cmp[cmp['model_abs_error'] > 2].sort_values('week', ascending=False)#.sort_values('model_abs_error', ascending=False)
big_misses[['season', 'week', 'player', 'position', 'team', 'actual_score',
            'model_predicted_score', 'espn_projection', 'model_error', 'espn_error', 'winner']].head(10)

Model: MAE=3.723  RMSE=5.150  R2=0.555
ESPN: MAE=3.752  RMSE=5.437  R2=0.504


,season,week,player,position,team,actual_score,model_predicted_score,espn_projection,model_error,espn_error,winner
120,2026,10,Nick Folk,K,NYJ,2.0,7.508949,7.622429,-5.508949,-5.622429,Model
22313,2026,10,Ashton Jeanty,RB,LV,9.4,14.614253,18.824199,-5.214253,-9.424199,Model
22428,2026,10,Dylan Sampson,RB,CLE,1.9,6.052701,4.627810,-4.152701,-2.727810,ESPN
18612,2026,10,John Metchie III,WR,NYJ,13.5,3.380778,2.005722,10.119222,11.494278,Model
10398,2026,10,Calvin Austin III,WR,PIT,1.5,8.508904,11.516949,-7.008904,-10.016949,Model
10574,2026,10,Christian Watson,WR,GB,20.6,8.575718,8.936682,12.024282,11.663318,ESPN
10631,2026,10,Matt Gay,K,WSH,5.0,7.800831,7.738991,-2.800831,-2.738991,ESPN
16466,2026,10,Michael Mayer,TE,LV,4.3,7.757628,8.022976,-3.457628,-3.722976,Model
16515,2026,10,Blake Corum,RB,LAR,1.5,5.413988,5.768150,-3.913988,-4.268150,Model
16548,2026,10,Arian Smith,WR,NYJ,0.0,3.265455,4.800053,-3.265455,-4.800053,Model


In [45]:
position_table = pd.crosstab(cmp['position'], cmp['winner'], margins=True)
print(position_table)

winner    ESPN  Model   All
position                   
K          117    176   293
QB         165    190   355
RB         543    418   961
TE         524    412   936
WR         798    730  1528
All       2147   1926  4073
